### Yearly performance comparison of RF and LSTM on (1) FI (2) TI (3) Big data, with Mann-Whitney tests to assess differences in prediction errors across datasets.

In [14]:
import setuptools
import distutils
import pandas as pd
import numpy as np
import os, random, time, warnings
import itertools
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.stats import skew, kurtosis, mannwhitneyu

warnings.filterwarnings('ignore')

torch.set_default_tensor_type(torch.FloatTensor)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

class LSTMNet(nn.Module):
    def __init__(self, i_size, h_size, n_layers, drop):
        super().__init__()
        self.lstm = nn.LSTM(i_size, h_size, n_layers, dropout=drop if n_layers > 1 else 0, batch_first=True)
        self.fc = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(drop), 
            nn.Linear(h_size, 1)
        )
        
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class Predictor:
    @staticmethod
    def get_rf_grid():
        return {
            'n_estimators': [200],
            'max_depth': [15],
            'min_samples_split': [10],
            'min_samples_leaf': [10],
            'max_features': [10]
        }

    @staticmethod
    def get_lstm_grid():
        return {
            'hidden_layers': [2],
            'hidden_nodes': [64],
            'lr': [0.005],
            'patience': [10],
            'bs': [32],
            'dropout': [0.2],
            'optim': ['Adam']
        }

    def train_rf(self, X_tr, y_tr, X_val, y_val, X_te):
        grid = self.get_rf_grid()
        keys, values = zip(*grid.items())
        param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
        
        best_mse, best_p = float('inf'), None
        
        for p in param_combinations:
            model = RandomForestRegressor(**p, random_state=42, n_jobs=-1)
            model.fit(X_tr, y_tr)
            mse = mean_squared_error(y_val, model.predict(X_val))
            if mse < best_mse: best_mse, best_p = mse, p
            
        final = RandomForestRegressor(**best_p, random_state=42, n_jobs=-1)
        final.fit(np.concatenate([X_tr, X_val]), np.concatenate([y_tr, y_val]))
        return final.predict(X_te)

    def train_lstm(self, X_tr, y_tr, X_val, y_val, X_te):
        scaler = StandardScaler()
        N, T, F = X_tr.shape
        X_tr = scaler.fit_transform(X_tr.reshape(-1, F)).reshape(N, T, F)
        X_val = scaler.transform(X_val.reshape(-1, F)).reshape(X_val.shape)
        X_te = scaler.transform(X_te.reshape(-1, F)).reshape(X_te.shape)
        
        Xt, yt = torch.FloatTensor(X_tr), torch.FloatTensor(y_tr)
        Xv, yv = torch.FloatTensor(X_val), torch.FloatTensor(y_val)
        Xte = torch.FloatTensor(X_te)
        
        def run_epoch(m, opt, loader):
            m.train()
            for x_b, y_b in loader:
                x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
                opt.zero_grad()
                nn.MSELoss()(m(x_b).squeeze(), y_b).backward()
                opt.step()

        grid = self.get_lstm_grid()
        keys, values = zip(*grid.items())
        param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
        
        best_loss, best_p = float('inf'), None
        
        for p in param_combinations:
            m = LSTMNet(F, p['hidden_nodes'], p['hidden_layers'], p['dropout']).to(DEVICE)
            opt = getattr(optim, p['optim'])(m.parameters(), lr=p['lr'])
            loader = DataLoader(TensorDataset(Xt, yt), batch_size=p['bs'], shuffle=False)
            
            min_v, pat = float('inf'), 0
            Xv_gpu, yv_gpu = Xv.to(DEVICE), yv.to(DEVICE)
            
            for _ in range(50):
                run_epoch(m, opt, loader)
                m.eval()
                with torch.no_grad():
                    v_loss = nn.MSELoss()(m(Xv_gpu).squeeze(), yv_gpu).item()
                if v_loss < min_v: min_v, pat = v_loss, 0
                else: pat += 1
                if pat >= p['patience']: break
            if min_v < best_loss: best_loss, best_p = min_v, p

        X_all, y_all = torch.cat([Xt, Xv]), torch.cat([yt, yv])
        split = int(len(X_all) * 0.8)
        loader = DataLoader(TensorDataset(X_all[:split], y_all[:split]), batch_size=best_p['bs'], shuffle=False)
        val_X, val_y = X_all[split:].to(DEVICE), y_all[split:].to(DEVICE)
        
        m = LSTMNet(F, best_p['hidden_nodes'], best_p['hidden_layers'], best_p['dropout']).to(DEVICE)
        opt = getattr(optim, p['optim'])(m.parameters(), lr=p['lr'])
        min_v, pat, best_sd = float('inf'), 0, None
        for _ in range(100):
            run_epoch(m, opt, loader)
            m.eval()
            with torch.no_grad():
                v_loss = nn.MSELoss()(m(val_X).squeeze(), val_y).item()
            if v_loss < min_v: min_v, pat, best_sd = v_loss, 0, m.state_dict()
            else: pat += 1
            if pat >= best_p['patience']: break
            
        if best_sd: m.load_state_dict(best_sd)
        
        preds = []
        m.eval()
        with torch.no_grad():
            for x_b, in DataLoader(TensorDataset(Xte), batch_size=best_p['bs'], shuffle=False):
                preds.extend(m(x_b.to(DEVICE)).squeeze().cpu().numpy())
        return np.array(preds)

class Pipeline:
    def __init__(self):
        self.pred = Predictor()
        self.dsets = {
            'FI': ('fundamental_data_72_features_lagged.csv', 'M', 12),
            'TI': ('technical_data_20_features.csv', 'D', 20),
            'BigData': ('integrated_data_92_features_lagged.csv', 'D', 20)
        }
        self.data, self.prices, self.stocks = {}, None, {}

    def load(self):
        p = pd.concat([pd.read_csv("TRD_Dalyr(20150105-20200103).csv"), pd.read_csv("TRD_Dalyr(20200106-20241213).csv")])
        p['code'] = p['Stkcd'].astype(str).str.zfill(6)
        p['date'] = pd.to_datetime(p['Trddt'])
        p = p.sort_values(['code', 'date']).reset_index(drop=True)
        p['d_ret'] = p.groupby('code')['Clsprc'].pct_change().fillna(0)
        m = p.groupby(['code', p['date'].dt.to_period('M').rename('ym')]).agg({'Clsprc':'last', 'date':'last'}).reset_index()
        m['m_ret'] = m.groupby('code')['Clsprc'].pct_change().fillna(0)
        self.prices = p.merge(m[['code', 'date', 'm_ret']], on=['code', 'date'], how='left')

        for y in range(2020, 2025):
            f = f'{y}年选择的股票列表.csv'
            if os.path.exists(f):
                self.stocks[y] = pd.read_csv(f)['Chameleon'].dropna().astype(int).astype(str).str.zfill(6).tolist()

        for k, (f, freq, _) in self.dsets.items():
            if os.path.exists(f):
                d = pd.read_csv(f)
                d['code'] = d['stock_code'].astype(str).str.zfill(6)
                d['date'] = pd.to_datetime(d['date'])
                
                if freq == 'M':
                    d['ym'] = d['date'].dt.to_period('M')
                    d = d.sort_values('date').groupby(['code', 'ym']).last().reset_index()
                    p_monthly = self.prices[['code', 'date', 'm_ret']].copy()
                    p_monthly['ym'] = p_monthly['date'].dt.to_period('M')
                    p_monthly = p_monthly.dropna(subset=['m_ret'])
                    d = d.drop(columns=['date']).merge(p_monthly[['code', 'ym', 'date', 'm_ret']], on=['code', 'ym'])
                    d = d.drop(columns=['ym'])
                    
                else:
                    tgt = self.prices[['code', 'date', 'd_ret']]
                    d = d.merge(tgt, on=['code', 'date'])
                
                self.data[k] = d.fillna(0).sort_values(['code', 'date'])

    def run(self):
        self.load()
        res = {'RF': [], 'LSTM': []}

        print("Stock preselection using the Chameleon:")
        for yr in range(2020, 2025):
            if yr in self.stocks:
                valid = [s for s in self.stocks[yr] if s in self.prices['code'].values]
                print(f"    {yr}年: {len(valid)}只股票")

        for yr in range(2020, 2025):
            if yr not in self.stocks: continue
            
            valid_stocks = [s for s in self.stocks[yr] if s in self.prices['code'].values]
            print(f"\n处理{yr}年 ({len(valid_stocks)}只股票)")
            
            train_ys, val_y, test_y = range(yr-5, yr-1), yr-1, yr
            
            for name in ['FI', 'TI', 'BigData']:
                _, _, lb = self.dsets[name]
                df = self.data[name]
                cols = [c for c in df.columns if c not in ['code','date','stock_code','m_ret','d_ret']]
                
                curr_valid_stocks = [s for s in valid_stocks if s in df['code'].values]
                
                for code in curr_valid_stocks:
                    sub = df[df['code'] == code]
                    if len(sub) < 50: continue
                    
                    feat, tgt = sub[cols].values, sub['m_ret' if name=='FI' else 'd_ret'].values
                    dts = sub['date'].values
                    
                    X_l, X_r, y, d_o, sk, ku = [], [], [], [], [], []
                    for i in range(lb, len(sub)):
                        w = tgt[i-lb:i]
                        X_l.append(feat[i-lb:i])
                        X_r.append(feat[i-lb:i].flatten())
                        y.append(tgt[i]); d_o.append(dts[i])
                        if len(w)>=5: sk.append(skew(w)); ku.append(kurtosis(w))
                        else: sk.append(0); ku.append(0)
                    
                    if not d_o: continue
                    
                    X_l, X_r, y = np.array(X_l), np.array(X_r), np.array(y)
                    d_o, sk, ku = np.array(d_o), np.array(sk), np.array(ku)
                    
                    yrs = pd.to_datetime(d_o).year
                    msk_tr = np.isin(yrs, train_ys)
                    msk_val = (yrs == val_y)
                    msk_te = (yrs == test_y)
                    
                    if not (msk_tr.any() and msk_val.any() and msk_te.any()): continue
                    
                    # 网格搜索训练
                    p_rf = self.pred.train_rf(X_r[msk_tr], y[msk_tr], X_r[msk_val], y[msk_val], X_r[msk_te])
                    p_lstm = self.pred.train_lstm(X_l[msk_tr], y[msk_tr], X_l[msk_val], y[msk_val], X_l[msk_te])
                    
                    print(f"{code}-{name}: {len(p_rf)}条RF记录, {len(p_lstm)}条LSTM记录")
                    
                    base = {'stock_code': code, 'year': yr, 'dataset': name}
                    for i, d in enumerate(d_o[msk_te]):
                        info = {**base, 'date': d, 'actual_return': y[msk_te][i], 'skew_20d': sk[msk_te][i], 'kurt_20d': ku[msk_te][i]}
                        res['RF'].append({**info, 'predicted_return': p_rf[i], 'prediction_error': y[msk_te][i] - p_rf[i]})
                        res['LSTM'].append({**info, 'predicted_return': p_lstm[i], 'prediction_error': y[msk_te][i] - p_lstm[i]})
        
        rf_df = pd.DataFrame(res['RF'])
        lstm_df = pd.DataFrame(res['LSTM'])
                
        print(f"\n预测完成:")
        print(f"  RF记录: {len(rf_df)}条")
        print(f"  LSTM记录: {len(lstm_df)}条")
        
        self.save_file(rf_df,lstm_df)
        self.performance_and_test(rf_df,lstm_df)
        
        
    def save_file(self,rf_df,lstm_df):
        for d in ['TI', 'FI', 'BigData']:
            sub = rf_df[rf_df['dataset'] == d].drop(columns=['year', 'dataset'])
            if not sub.empty: sub.to_csv(f"{d}_RF_predictions.csv", index=False, encoding='utf-8-sig')
            sub = lstm_df[lstm_df['dataset'] == d].drop(columns=['year', 'dataset'])
            if not sub.empty: sub.to_csv(f"{d}_LSTM_predictions.csv", index=False, encoding='utf-8-sig')
    
    def performance_and_test(self, rf_df,lstm_df):
        print("\n  RF误差统计:")
        for d in ['FI', 'TI', 'BigData']:
            sub = rf_df[rf_df['dataset'] == d]
            ae_values = np.abs(sub['actual_return'] - sub['predicted_return'])
            se_values = (sub['actual_return'] - sub['predicted_return']) ** 2
            ae_len = len(ae_values)
            se_len = len(se_values)
            print(f"    {d}: AE length={ae_len}, SE length={se_len}")
            
        print("  LSTM误差统计:")
        for d in ['FI', 'TI', 'BigData']:
            sub = lstm_df[lstm_df['dataset'] == d]
            ae_values = np.abs(sub['actual_return'] - sub['predicted_return'])
            se_values = (sub['actual_return'] - sub['predicted_return']) ** 2
            ae_len = len(ae_values)
            se_len = len(se_values)
            print(f"    {d}: AE length={ae_len}, SE length={se_len}")

        for model_name, df in [('RF', rf_df), ('LSTM', lstm_df)]:
            model_stats = []
            for y in df['year'].unique():
                for d in df['dataset'].unique():
                    subset = df[(df['year'] == y) & (df['dataset'] == d)]
                    stock_maes, stock_mapes, stock_mses, stock_rmses, stock_u1s, stock_hrs = [], [], [], [], [], []
                    
                    for code, stock_data in subset.groupby('stock_code'):
                        a = stock_data['actual_return'].values
                        p = stock_data['predicted_return'].values

                        mse = mean_squared_error(a, p)
                        rmse = np.sqrt(mse)
                        denom = np.sqrt(np.mean(a**2)) + np.sqrt(np.mean(p**2))
                        
                        stock_maes.append(mean_absolute_error(a, p))
                        stock_mses.append(mse)
                        stock_rmses.append(rmse)
                        stock_u1s.append(rmse / denom if denom != 0 else np.nan)

                        non_zero_mask = a != 0
                        if non_zero_mask.sum() > 0:
                            mape = np.mean(np.abs((a[non_zero_mask] - p[non_zero_mask]) / a[non_zero_mask])) * 100
                            stock_mapes.append(mape)
                        else:
                            stock_mapes.append(np.nan)
                        
                        pos_pred = p > 0
                        if pos_pred.sum() > 0:
                            stock_hrs.append(((a > 0) & (p > 0)).sum() / pos_pred.sum())
                        else:
                            stock_hrs.append(np.nan)

                    d_display = 'Big data' if d == 'BigData' else d
                    model_stats.append({
                        'Year': y, 'Data': d_display, 
                        'MAE': np.nanmean(stock_maes), 
                        'MAPE': np.nanmean(stock_mapes), 
                        'MSE': np.nanmean(stock_mses), 
                        'RMSE': np.nanmean(stock_rmses), 
                        'U1': np.nanmean(stock_u1s), 
                        'HR+': np.nanmean(stock_hrs)
                    })
            
            big = df[df['dataset']=='BigData']
            for t in ['FI', 'TI']:
                oth = df[df['dataset']==t]
                if not big.empty and not oth.empty:
                    try:
                        # 1. AE Test (MAE)
                        ae_big = np.abs(big['actual_return'] - big['predicted_return'])
                        ae_oth = np.abs(oth['actual_return'] - oth['predicted_return'])
                        _, p_mae = mannwhitneyu(ae_big, ae_oth, alternative='two-sided')
                        
                        # 2. SE Test (MSE)
                        se_big = (big['actual_return'] - big['predicted_return']) ** 2
                        se_oth = (oth['actual_return'] - oth['predicted_return']) ** 2
                        _, p_mse = mannwhitneyu(se_big, se_oth, alternative='two-sided')
                        
                        model_stats.append({
                            'Year': 'Total', 'Data': f'MW(Big vs {t})', 
                            'MAE': p_mae, 
                            'MAPE': '',
                            'MSE': p_mse, 
                            'RMSE':'', 'U1':'', 'HR+':''
                        })
                    except: pass
            pd.DataFrame(model_stats).to_csv(f"{model_name}_gpu_accelerated.csv", index=False, encoding='utf-8-sig')

if __name__ == "__main__":
    t0 = time.time()
    pipe = Pipeline()
    pipe.run()
    
    print("\n生成的文件:")
    files = ["TI_RF_predictions.csv", "TI_LSTM_predictions.csv", "FI_RF_predictions.csv", "FI_LSTM_predictions.csv",
             "BigData_RF_predictions.csv", "BigData_LSTM_predictions.csv", "RF_gpu_accelerated.csv", "LSTM_gpu_accelerated.csv"]
    for i, f in enumerate(files, 1):
        if os.path.exists(f):
            print(f"{i}. {f}")
            
    total_time = (time.time()-t0)/60
    print(f"\n总耗时: {total_time:.2f} min")

Stock preselection using the Chameleon:
    2020年: 16只股票
    2021年: 15只股票
    2022年: 22只股票
    2023年: 21只股票
    2024年: 16只股票

处理2020年 (16只股票)
300059-FI: 12条RF记录, 12条LSTM记录
002601-FI: 12条RF记录, 12条LSTM记录
000568-FI: 12条RF记录, 12条LSTM记录
600176-FI: 12条RF记录, 12条LSTM记录
000625-FI: 12条RF记录, 12条LSTM记录
600196-FI: 12条RF记录, 12条LSTM记录
600760-FI: 12条RF记录, 12条LSTM记录
000768-FI: 12条RF记录, 12条LSTM记录
002304-FI: 12条RF记录, 12条LSTM记录
601899-FI: 12条RF记录, 12条LSTM记录
002352-FI: 12条RF记录, 12条LSTM记录
600893-FI: 12条RF记录, 12条LSTM记录
601919-FI: 12条RF记录, 12条LSTM记录
600436-FI: 12条RF记录, 12条LSTM记录
002493-FI: 12条RF记录, 12条LSTM记录
300015-FI: 12条RF记录, 12条LSTM记录
300059-TI: 243条RF记录, 243条LSTM记录
002601-TI: 243条RF记录, 243条LSTM记录
000568-TI: 243条RF记录, 243条LSTM记录
600176-TI: 233条RF记录, 233条LSTM记录
000625-TI: 243条RF记录, 243条LSTM记录
600196-TI: 243条RF记录, 243条LSTM记录
600760-TI: 243条RF记录, 243条LSTM记录
000768-TI: 243条RF记录, 243条LSTM记录
002304-TI: 243条RF记录, 243条LSTM记录
601899-TI: 243条RF记录, 243条LSTM记录
002352-TI: 243条RF记录, 243条LSTM记录
600893-TI: 242条RF记录, 242条L

RF预测结果：

In [3]:
file1 = "RF_gpu_accelerated.csv"
df1 = pd.read_csv(file1)
df1

,Year,Data,MAE,MAPE,MSE,RMSE,U1,HR+
0,2020,FI,1.111409e-01,96.892144,2.161081e-02,0.147006,0.915520,0.591398
1,2020,TI,7.228750e-02,276.501259,8.731105e-03,0.093440,0.785550,0.528969
2,2020,Big data,2.578264e-02,108.088899,1.197885e-03,0.034610,0.746225,0.531980
3,2021,FI,9.682517e-02,95.791971,1.595954e-02,0.126331,0.894247,0.597561
4,2021,TI,7.369518e-02,274.900910,9.292724e-03,0.096399,0.785668,0.511069
5,2021,Big data,2.573192e-02,107.258736,1.180921e-03,0.034365,0.746929,0.492974
6,2022,FI,8.565037e-02,99.497990,1.309064e-02,0.114414,0.890367,0.507812
7,2022,TI,5.813831e-02,260.075979,5.678890e-03,0.075358,0.776916,0.488976
8,2022,Big data,2.255377e-02,111.694531,8.681113e-04,0.029464,0.731571,0.469460
9,2023,FI,6.022134e-02,99.833265,7.139340e-03,0.084495,0.854443,0.587413


LSTM预测结果：

In [4]:
file2 = "LSTM_gpu_accelerated.csv"
df2 = pd.read_csv(file2)
df2

,Year,Data,MAE,MAPE,MSE,RMSE,U1,HR+
0,2020,FI,1.320318e-01,140.819592,2.974073e-02,0.172455,0.707516,0.605769
1,2020,TI,2.613978e-02,109.729600,1.278061e-03,0.035750,0.728098,0.517641
2,2020,Big data,2.286534e-02,99.941777,1.010791e-03,0.031793,0.881576,0.535758
3,2021,FI,1.122384e-01,130.056512,1.996335e-02,0.141292,0.721874,0.531646
4,2021,TI,2.393025e-02,103.311695,1.058729e-03,0.032538,0.780944,0.484688
5,2021,Big data,2.252322e-02,99.397463,9.687233e-04,0.031124,0.876916,0.500000
6,2022,FI,9.946790e-02,137.469882,1.833590e-02,0.135410,0.739442,0.574803
7,2022,TI,2.535679e-02,113.919926,2.021026e-03,0.044956,0.713238,0.480763
8,2022,Big data,1.908643e-02,100.241603,6.712230e-04,0.025908,0.845001,0.481322
9,2023,FI,7.582115e-02,140.540996,1.171064e-02,0.108216,0.681745,0.593985


### Performance comparison with various popular predictive models.

In [15]:
import pandas as pd
import numpy as np
import os, random, time, warnings
import itertools
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.svm import SVR
from sklearn.metrics import  mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.stats import skew, kurtosis

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')

torch.set_default_tensor_type(torch.FloatTensor)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

class CNNNet(nn.Module):
    def __init__(self, i_size, h_size, seq_len, drop):
        super().__init__()
        self.conv1 = nn.Conv1d(i_size, h_size, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(h_size, h_size*2, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2) 
        self.dropout = nn.Dropout(drop)
        conv_out_len = seq_len // 2 // 2 

        self.fc = nn.Sequential(
            nn.Linear(h_size * 2 * conv_out_len, h_size),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(h_size, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

class RNNNet(nn.Module):
    def __init__(self, i_size, h_size, n_layers, drop):
        super().__init__()
        self.rnn = nn.RNN(i_size, h_size, n_layers, dropout=drop if n_layers > 1 else 0, batch_first=True)
        self.fc = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(h_size, 1)
        )
        
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

class Predictor:
    @staticmethod
    def get_arima_grid():
        return {
            'p': [1],
            'd': [1],
            'q': [1]
        }

    @staticmethod
    def get_svr_grid():
        return {
            'C': [10.0],
            'epsilon': [0.1],
            'gamma': ['scale']
        }

    @staticmethod
    def get_nn_grid():
        # CNN 和 RNN
        return {
            'hidden_nodes': [64],
            'lr': [0.005],
            'patience': [10],
            'bs': [32],
            'dropout': [0.2],
            'optim': ['Adam']
        }

    def train_arima(self, y_tr, y_val, steps_te):
        history = list(y_tr)
        grid = self.get_arima_grid()
        best_aic, best_order = float('inf'), (1, 1, 1)
        d_val = 0
        try:
            if adfuller(history)[1] > 0.05: d_val = 1
        except: pass
        
        for p in grid['p']:
            for q in grid['q']:
                try:
                    model = ARIMA(history, order=(p, d_val, q))
                    res = model.fit()
                    if res.aic < best_aic: best_aic, best_order = res.aic, (p, d_val, q)
                except: continue

        full_hist = list(np.concatenate([y_tr, y_val]))
        try:
            model = ARIMA(full_hist, order=best_order)
            model_fit = model.fit()
            forecast = model_fit.forecast(steps=steps_te)
            return np.array(forecast)
        except:
            return np.zeros(steps_te)

    def train_svr(self, X_tr, y_tr, X_val, y_val, X_te):
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)
        X_te = scaler.transform(X_te)

        grid = self.get_svr_grid()
        keys, values = zip(*grid.items())
        param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
        
        best_mse, best_p = float('inf'), None
        
        for p in param_combinations:
            model = SVR(**p)
            model.fit(X_tr, y_tr)
            mse = mean_squared_error(y_val, model.predict(X_val))
            if mse < best_mse: best_mse, best_p = mse, p
            
        final = SVR(**best_p)
        final.fit(np.concatenate([X_tr, X_val]), np.concatenate([y_tr, y_val]))
        return final.predict(X_te)

    def _train_torch_model(self, ModelClass, X_tr, y_tr, X_val, y_val, X_te, is_cnn=False):
        scaler = StandardScaler()
        N, T, F = X_tr.shape
        X_tr = scaler.fit_transform(X_tr.reshape(-1, F)).reshape(N, T, F)
        X_val = scaler.transform(X_val.reshape(-1, F)).reshape(X_val.shape)
        X_te = scaler.transform(X_te.reshape(-1, F)).reshape(X_te.shape)
        
        Xt, yt = torch.FloatTensor(X_tr), torch.FloatTensor(y_tr)
        Xv, yv = torch.FloatTensor(X_val), torch.FloatTensor(y_val)
        Xte = torch.FloatTensor(X_te)
        
        def run_epoch(m, opt, loader):
            m.train()
            for x_b, y_b in loader:
                x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
                opt.zero_grad()
                nn.MSELoss()(m(x_b).squeeze(), y_b).backward()
                opt.step()

        grid = self.get_nn_grid()
        keys, values = zip(*grid.items())
        param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
        
        best_loss, best_p = float('inf'), None
        
        for p in param_combinations:
            if is_cnn:
                m = ModelClass(F, p['hidden_nodes'], T, p['dropout']).to(DEVICE)
            else: # RNN
                m = ModelClass(F, p['hidden_nodes'], 2, p['dropout']).to(DEVICE)
                
            opt = getattr(optim, p['optim'])(m.parameters(), lr=p['lr'])
            loader = DataLoader(TensorDataset(Xt, yt), batch_size=p['bs'], shuffle=False)
            
            min_v, pat = float('inf'), 0
            Xv_gpu, yv_gpu = Xv.to(DEVICE), yv.to(DEVICE)
            
            for _ in range(50):
                run_epoch(m, opt, loader)
                m.eval()
                with torch.no_grad():
                    v_loss = nn.MSELoss()(m(Xv_gpu).squeeze(), yv_gpu).item()
                if v_loss < min_v: min_v, pat = v_loss, 0
                else: pat += 1
                if pat >= p['patience']: break
            if min_v < best_loss: best_loss, best_p = min_v, p

        # 使用最佳参数重新训练
        X_all, y_all = torch.cat([Xt, Xv]), torch.cat([yt, yv])
        split = int(len(X_all) * 0.8)
        loader = DataLoader(TensorDataset(X_all[:split], y_all[:split]), batch_size=best_p['bs'], shuffle=False)
        val_X, val_y = X_all[split:].to(DEVICE), y_all[split:].to(DEVICE)
        
        if is_cnn:
            m = ModelClass(F, best_p['hidden_nodes'], T, best_p['dropout']).to(DEVICE)
        else:
            m = ModelClass(F, best_p['hidden_nodes'], 2, best_p['dropout']).to(DEVICE)
            
        opt = getattr(optim, best_p['optim'])(m.parameters(), lr=best_p['lr'])
        min_v, pat, best_sd = float('inf'), 0, None
        
        for _ in range(100):
            run_epoch(m, opt, loader)
            m.eval()
            with torch.no_grad():
                v_loss = nn.MSELoss()(m(val_X).squeeze(), val_y).item()
            if v_loss < min_v: min_v, pat, best_sd = v_loss, 0, m.state_dict()
            else: pat += 1
            if pat >= best_p['patience']: break
            
        if best_sd: m.load_state_dict(best_sd)
        
        preds = []
        m.eval()
        with torch.no_grad():
            for x_b, in DataLoader(TensorDataset(Xte), batch_size=best_p['bs'], shuffle=False):
                preds.extend(m(x_b.to(DEVICE)).squeeze().cpu().numpy())
        return np.array(preds)

    def train_cnn(self, X_tr, y_tr, X_val, y_val, X_te):
        return self._train_torch_model(CNNNet, X_tr, y_tr, X_val, y_val, X_te, is_cnn=True)

    def train_rnn(self, X_tr, y_tr, X_val, y_val, X_te):
        return self._train_torch_model(RNNNet, X_tr, y_tr, X_val, y_val, X_te, is_cnn=False)

class Pipeline:
    def __init__(self):
        self.pred = Predictor()
        self.dsets = {
            'BigData': ('integrated_data_92_features_lagged.csv', 'D', 20)
        }
        self.data, self.prices, self.stocks = {}, None, {}

    def load(self):
        p = pd.concat([pd.read_csv("TRD_Dalyr(20150105-20200103).csv"), pd.read_csv("TRD_Dalyr(20200106-20241213).csv")])
        p['code'] = p['Stkcd'].astype(str).str.zfill(6)
        p['date'] = pd.to_datetime(p['Trddt'])
        p = p.sort_values(['code', 'date']).reset_index(drop=True)
        p['d_ret'] = p.groupby('code')['Clsprc'].pct_change().fillna(0)
        self.prices = p[['code', 'date', 'd_ret']]
        for y in range(2020, 2025):
            f = f'{y}年选择的股票列表.csv' 
            if os.path.exists(f):
                try:
                    df_stk = pd.read_csv(f)
                    col = 'Chameleon' if 'Chameleon' in df_stk.columns else df_stk.columns[0]
                    self.stocks[y] = df_stk[col].dropna().astype(int).astype(str).str.zfill(6).tolist()
                except: pass

        for k, (f, freq, _) in self.dsets.items():
            if os.path.exists(f):
                d = pd.read_csv(f)
                d['code'] = d['stock_code'].astype(str).str.zfill(6)
                d['date'] = pd.to_datetime(d['date'])
                
                # 合并收益率
                d = d.merge(self.prices, on=['code', 'date'])
                self.data[k] = d.fillna(0).sort_values(['code', 'date'])

    def run(self):
        self.load()
        res = {'ARIMA': [], 'SVR': [], 'CNN': [], 'RNN': []}

        print("Stock preselection using the Chameleon:")
        for yr in range(2020, 2025):
            if yr in self.stocks:
                valid = [s for s in self.stocks[yr] if s in self.prices['code'].values]
                print(f"    {yr}年: {len(valid)}只股票")

        for yr in range(2020, 2025):
            valid_stocks = [s for s in self.stocks[yr] if s in self.prices['code'].values]
            print(f"\n处理{yr}年 ({len(valid_stocks)}只股票)")
            
            train_ys, val_y, test_y = range(yr-5, yr-1), yr-1, yr
            
            name = 'BigData'
            _, _, lb = self.dsets[name]
            df = self.data[name]
            cols = [c for c in df.columns if c not in ['code','date','stock_code','d_ret']]
            
            curr_valid_stocks = [s for s in valid_stocks if s in df['code'].values]
            
            for code in curr_valid_stocks:
                sub = df[df['code'] == code]
                if len(sub) < 50: continue
                
                feat, tgt = sub[cols].values, sub['d_ret'].values
                dts = sub['date'].values
                
                X_l, X_r, y, d_o, sk, ku = [], [], [], [], [], []
                for i in range(lb, len(sub)):
                    w = tgt[i-lb:i]
                    X_l.append(feat[i-lb:i])
                    X_r.append(feat[i-lb:i].flatten())
                    y.append(tgt[i]); d_o.append(dts[i])
                    if len(w)>=5: sk.append(skew(w)); ku.append(kurtosis(w))
                    else: sk.append(0); ku.append(0)
                
                if not d_o: continue
                
                X_l, X_r, y = np.array(X_l), np.array(X_r), np.array(y)
                d_o, sk, ku = np.array(d_o), np.array(sk), np.array(ku)
                
                yrs = pd.to_datetime(d_o).year
                msk_tr = np.isin(yrs, train_ys)
                msk_val = (yrs == val_y)
                msk_te = (yrs == test_y)
                
                if not (msk_tr.any() and msk_val.any() and msk_te.any()): continue

                p_arima = self.pred.train_arima(y[msk_tr], y[msk_val], len(y[msk_te]))
                p_svr = self.pred.train_svr(X_r[msk_tr], y[msk_tr], X_r[msk_val], y[msk_val], X_r[msk_te])
                p_cnn = self.pred.train_cnn(X_l[msk_tr], y[msk_tr], X_l[msk_val], y[msk_val], X_l[msk_te])
                p_rnn = self.pred.train_rnn(X_l[msk_tr], y[msk_tr], X_l[msk_val], y[msk_val], X_l[msk_te])
                
                print(f"{code}: ARIMA|SVR|CNN|RNN 预测完成")
                
                base = {'stock_code': code}
                for i, d in enumerate(d_o[msk_te]):
                    if i >= len(p_arima): break
                    info = {**base, 'date': d, 'actual_return': y[msk_te][i], 'skew_20d': sk[msk_te][i], 'kurt_20d': ku[msk_te][i]}
                    res['ARIMA'].append({**info, 'predicted_return': p_arima[i], 'prediction_error': y[msk_te][i] - p_arima[i]})
                    res['SVR'].append({**info, 'predicted_return': p_svr[i], 'prediction_error': y[msk_te][i] - p_svr[i]})
                    res['CNN'].append({**info, 'predicted_return': p_cnn[i], 'prediction_error': y[msk_te][i] - p_cnn[i]})
                    res['RNN'].append({**info, 'predicted_return': p_rnn[i], 'prediction_error': y[msk_te][i] - p_rnn[i]})
                        
        self.save_file(res)

    def save_file(self, res):
        dfs = {k: pd.DataFrame(v) for k, v in res.items()}

        for name, df in dfs.items():
            if not df.empty:
                df.to_csv(f"BigData_{name}_predictions.csv", index=False, encoding='utf-8-sig')

        print(f"\n预测完成 (BigData):")
        for name, df in dfs.items():
            print(f"  {name}记录: {len(df)}条")

if __name__ == "__main__":
    t0 = time.time()
    pipe = Pipeline()
    pipe.run()
    
    print("\n生成的预测文件:")
    files = ["BigData_ARIMA_predictions.csv", "BigData_SVR_predictions.csv", 
             "BigData_CNN_predictions.csv", "BigData_RNN_predictions.csv"]
    for i, f in enumerate(files, 1):
        if os.path.exists(f):
            print(f"{i}. {f}")
            
    total_time = (time.time()-t0)/60
    print(f"\n总耗时: {total_time:.2f} min")

Stock preselection using the Chameleon:
    2020年: 16只股票
    2021年: 15只股票
    2022年: 22只股票
    2023年: 21只股票
    2024年: 16只股票

处理2020年 (16只股票)
300059: ARIMA|SVR|CNN|RNN 预测完成
002601: ARIMA|SVR|CNN|RNN 预测完成
000568: ARIMA|SVR|CNN|RNN 预测完成
600176: ARIMA|SVR|CNN|RNN 预测完成
000625: ARIMA|SVR|CNN|RNN 预测完成
600196: ARIMA|SVR|CNN|RNN 预测完成
600760: ARIMA|SVR|CNN|RNN 预测完成
000768: ARIMA|SVR|CNN|RNN 预测完成
002304: ARIMA|SVR|CNN|RNN 预测完成
601899: ARIMA|SVR|CNN|RNN 预测完成
002352: ARIMA|SVR|CNN|RNN 预测完成
600893: ARIMA|SVR|CNN|RNN 预测完成
601919: ARIMA|SVR|CNN|RNN 预测完成
600436: ARIMA|SVR|CNN|RNN 预测完成
002493: ARIMA|SVR|CNN|RNN 预测完成
300015: ARIMA|SVR|CNN|RNN 预测完成

处理2021年 (15只股票)
300033: ARIMA|SVR|CNN|RNN 预测完成
601618: ARIMA|SVR|CNN|RNN 预测完成
300059: ARIMA|SVR|CNN|RNN 预测完成
002179: ARIMA|SVR|CNN|RNN 预测完成
601225: ARIMA|SVR|CNN|RNN 预测完成
601238: ARIMA|SVR|CNN|RNN 预测完成
002230: ARIMA|SVR|CNN|RNN 预测完成
002241: ARIMA|SVR|CNN|RNN 预测完成
601808: ARIMA|SVR|CNN|RNN 预测完成
002271: ARIMA|SVR|CNN|RNN 预测完成
000776: ARIMA|SVR|CNN|RNN 预测完成
6018

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

def calculate_stock_metrics(stock_df):
    """计算单只股票的指标值"""
    a = stock_df['actual_return'].values
    p = stock_df['predicted_return'].values
    mask = np.isfinite(a) & np.isfinite(p)
    a, p = a[mask], p[mask]
    # MAE
    mae = mean_absolute_error(a, p)
    # MSE & RMSE
    mse = mean_squared_error(a, p)
    rmse = np.sqrt(mse)
    # U1
    denom = np.sqrt(np.mean(a**2)) + np.sqrt(np.mean(p**2))
    u1 = rmse / denom if denom != 0 else np.nan
    # MAPE
    non_zero_mask = a != 0
    if non_zero_mask.sum() > 0:
        mape = np.mean(np.abs((a[non_zero_mask] - p[non_zero_mask]) / a[non_zero_mask])) * 100
    else:
        mape = np.nan
    # HR+
    pos_pred = p > 0
    if pos_pred.sum() > 0:
        hr_plus = ((a > 0) & (p > 0)).sum() / pos_pred.sum()
    else:
        hr_plus = np.nan
    
    return {
        'MAE': mae,
        'MAPE': mape,
        'MSE': mse,
        'RMSE': rmse,
        'U1': u1,
        'HR+': hr_plus
    }

def clean_data(df):
    """清理数据"""
    # 移除缺失值
    df = df.dropna(subset=['actual_return', 'predicted_return'])
    
    # 移除无穷大值
    df = df[~np.isinf(df['actual_return'])]
    df = df[~np.isinf(df['predicted_return'])]
    
    # 移除极端异常值 (超过±100%的收益率)
    df = df[(np.abs(df['actual_return']) <= 1.0) & 
            (np.abs(df['predicted_return']) <= 1.0)]
    
    return df

def process_model(model_name, file_path):
    """处理单个模型的数据"""
    df = pd.read_csv(file_path)
    df_clean = clean_data(df)
    stock_codes = df_clean['stock_code'].unique()
    
    # 存储每只股票的指标
    all_stock_metrics = {
        'MAE': [],
        'MAPE': [],
        'MSE': [],
        'RMSE': [],
        'U1': [],
        'HR+': []
    }
    
    # 对每只股票计算指标
    for stock_code in stock_codes:
        stock_df = df_clean[df_clean['stock_code'] == stock_code]
        stock_metrics = calculate_stock_metrics(stock_df)
        
        # 将该股票的指标添加到列表中
        for metric in ['MAE', 'MAPE', 'MSE', 'RMSE', 'U1', 'HR+']:
            if not np.isnan(stock_metrics[metric]):
                all_stock_metrics[metric].append(stock_metrics[metric])
    
    # 计算所有股票的mean和SD
    metrics_stats = {}
    for metric in ['MAE', 'MAPE', 'MSE', 'RMSE', 'U1', 'HR+']:
        mean_val = np.mean(all_stock_metrics[metric])
        sd_val = np.std(all_stock_metrics[metric], ddof=1)
        metrics_stats[metric] = {'mean': mean_val, 'sd': sd_val}
    
    return metrics_stats

def main():
    models = {
        'RNN': 'BigData_RNN_predictions.csv',
        'CNN': 'BigData_CNN_predictions.csv',
        'SVR': 'BigData_SVR_predictions.csv',
        'ARIMA': 'BigData_ARIMA_predictions.csv',
        'RF': 'BigData_RF_predictions.csv',
        'LSTM': 'BigData_LSTM_predictions.csv'
    }
    
    # 存储所有结果
    all_results = {}
    
    # 处理每个模型
    for model_name, file_path in models.items():
        result = process_model(model_name, file_path)
        if result is not None:
            all_results[model_name] = result
    
    # 生成统计表
    if len(all_results) > 0:
        comparative_data = []
        
        # 为每个模型生成mean和SD行
        for model_name in ['ARIMA', 'SVR', 'RF', 'CNN', 'RNN', 'LSTM']:
            if model_name in all_results:
                stats = all_results[model_name]
                
                # mean行
                comparative_data.append({
                    'Model': model_name,
                    'Statistics': 'mean',
                    'MAE': stats['MAE']['mean'],
                    'MAPE': stats['MAPE']['mean'],
                    'MSE': stats['MSE']['mean'],
                    'RMSE': stats['RMSE']['mean'],
                    'U1': stats['U1']['mean'],
                    'HR+': stats['HR+']['mean']
                })
                
                # SD行
                comparative_data.append({
                    'Model': model_name,
                    'Statistics': 'SD',
                    'MAE': stats['MAE']['sd'],
                    'MAPE': stats['MAPE']['sd'],
                    'MSE': stats['MSE']['sd'],
                    'RMSE': stats['RMSE']['sd'],
                    'U1': stats['U1']['sd'],
                    'HR+': stats['HR+']['sd']
                })
        
        # 创建DataFrame
        comparative_df = pd.DataFrame(comparative_data)
        
        # 数值格式化
        numeric_cols = ['MAE', 'MAPE', 'MSE', 'RMSE', 'U1', 'HR+']
        for col in numeric_cols:
            comparative_df[col] = comparative_df[col].apply( 
                lambda x: f"{x:.6f}" if isinstance(x, (int, float)) and not np.isnan(x) else x
            )
        print(comparative_df.to_string(index=False))
        output_file = "最终结果.csv"
        comparative_df.to_csv(output_file, index=False, encoding='utf-8-sig')

if __name__ == "__main__":
    main()

Model Statistics      MAE       MAPE      MSE     RMSE       U1      HR+
ARIMA       mean 0.032929 163.539892 0.001446 0.037802 0.732584 0.495776
ARIMA         SD 0.001913  13.511351 0.000187 0.004174 0.035091 0.028126
  SVR       mean 0.023700 119.481458 0.001001 0.030424 0.707243 0.472222
  SVR         SD 0.002086  39.056753 0.000192 0.008726 0.046847 0.335777
   RF       mean 0.021520 111.943244 0.000833 0.028302 0.740238 0.490362
   RF         SD 0.001928   6.757944 0.000168 0.005720 0.019966 0.040459
  CNN       mean 0.023232 113.446142 0.000962 0.030217 0.704766 0.473708
  CNN         SD 0.002029  16.626983 0.000192 0.007390 0.037151 0.251526
  RNN       mean 0.024665 121.141300 0.000938 0.030130 0.714400 0.495548
  RNN         SD 0.001822  16.252225 0.000158 0.005015 0.025868 0.027797
 LSTM       mean 0.017529 100.281857 0.000629 0.024095 0.813990 0.508172
 LSTM         SD 0.001807   3.147329 0.000152 0.007006 0.059220 0.052870


### Diebold-Mariano test to evaluate performance differences between predictive models.

DM检验——MSE

In [13]:
import pandas as pd
import numpy as np
from scipy import stats

models = ['ARIMA', 'SVR', 'RF', 'CNN', 'RNN', 'LSTM']

data_dict = {}
for model in models:
    filename = f'BigData_{model}_predictions.csv'
    df = pd.read_csv(filename)
    data_dict[model] = df
sample_df = data_dict[models[0]]
identifier_cols = ['date']

stock_cols = ['stock_code']
for col in stock_cols:
    if col in sample_df.columns:
        identifier_cols.append(col)
        break


for model in models:
    df = data_dict[model]
    df['_merge_key'] = df[identifier_cols].astype(str).agg('_'.join, axis=1)
    data_dict[model] = df

# 获取第一个模型的键集合
common_keys = set(data_dict[models[0]]['_merge_key'])

# 与其他模型取交集
for model in models[1:]:
    model_keys = set(data_dict[model]['_merge_key'])
    common_keys = common_keys.intersection(model_keys)

# 转换为列表
common_keys = list(common_keys)
aligned_data_dict = {}
errors_dict = {}

for model in models:
    # 筛选出共同键的数据
    df_aligned = data_dict[model][data_dict[model]['_merge_key'].isin(common_keys)].copy()
    
    # 按标识列排序以确保顺序一致
    df_aligned = df_aligned.sort_values(identifier_cols).reset_index(drop=True)
    
    # 存储对齐后的数据
    aligned_data_dict[model] = df_aligned
    
    # 计算误差
    errors_dict[model] = (df_aligned['actual_return'] - df_aligned['predicted_return']).values

# Diebold-Mariano检验函数
def dm_test(e1, e2, h=1, power=2):
    """
    Diebold-Mariano检验
    
    参数:
    e1: 模型1的预测误差
    e2: 模型2的预测误差
    h: 预测步长 (默认为1)
    power: 损失函数的幂次 (对于MSE使用2)
    
    返回:
    DM统计量和p值
    """
    # 计算损失差异
    d = e1**power - e2**power
    
    # 计算均值
    d_mean = np.mean(d)
    
    # 计算方差（考虑自相关）
    def autocovariance(series, lag):
        n = len(series)
        c0 = np.dot(series, series) / n
        if lag == 0:
            return c0
        else:
            c_lag = np.dot(series[:-lag], series[lag:]) / n
            return c_lag
    
    # 计算长期方差
    gamma_0 = autocovariance(d, 0)
    gamma_sum = 0
    for lag in range(1, h):
        gamma_sum += autocovariance(d, lag)
    
    d_var = (gamma_0 + 2 * gamma_sum) / len(d)
    
    # 避免除以零
    if d_var <= 0:
        return np.nan, np.nan
    
    # 计算DM统计量
    dm_stat = d_mean / np.sqrt(d_var)
    
    # 计算p值（双侧检验）
    p_value = 2 * (1 - stats.norm.cdf(np.abs(dm_stat)))
    
    return dm_stat, p_value

# 存储原始p值用于后续分析
p_values = {}
dm_stats = {}

# 对所有模型对进行DM检验
for i, model1 in enumerate(models):
    for j, model2 in enumerate(models):
        if i < j:
            # 执行DM检验（使用绝对误差，power=2）
            dm_stat, p_value = dm_test(errors_dict[model1], errors_dict[model2], power=2)
            
            # 存储原始值
            p_values[(model1, model2)] = p_value
            dm_stats[(model1, model2)] = dm_stat

print("=" * 70)
print("Diebold-Mariano检验结果表 (DM p-value):")
print("=" * 70)
print()

detailed_results = []
for i, model1 in enumerate(models):
    for j, model2 in enumerate(models):
        if i < j:
            p_value = p_values[(model1, model2)]
            dm_stat = dm_stats[(model1, model2)]

            if p_value < 0.01:
                significance = "**"
            elif p_value < 0.05:
                significance = "*"
            
            # 判断哪个模型更好和结论
            if p_value < 0.05:  # 拒绝原假设
                if dm_stat > 0:  # Model1的MSE > Model2的MSE
                    better_model = model2
                else:  # Model1的MSE < Model2的MSE
                    better_model = model1
            else:  # 不能拒绝原假设
                better_model = "无显著差异"
            
            detailed_results.append({
                'Model 1': model1,
                'Model 2': model2,
                'DM Statistic': dm_stat,
                'p-value': p_value,
                'Significance': significance,
                'Better Model': better_model
            })

detailed_df = pd.DataFrame(detailed_results)

# 打印详细结果摘要
print(f"{'比较':13s} {'DM统计量':>12s} {'p值':>10s} {'结论':>15s}")
print("-" * 95)
for _, row in detailed_df.iterrows():
    comparison = f"{row['Model 1']} vs {row['Model 2']}"
    better = row['Better Model']
    if better != '无显著差异':
        better_mark = f"{better}{row['Significance']}"
    else:
        better_mark = better
    print(f"{comparison:15s} {row['DM Statistic']:12.4f} {row['p-value']:10.6f} {better_mark:>15s}")

Diebold-Mariano检验结果表 (DM p-value):

比较                   DM统计量         p值              结论
-----------------------------------------------------------------------------------------------
ARIMA vs SVR         27.5357   0.000000           SVR**
ARIMA vs RF          49.1377   0.000000            RF**
ARIMA vs CNN         32.0167   0.000000           CNN**
ARIMA vs RNN         77.5980   0.000000           RNN**
ARIMA vs LSTM        66.5245   0.000000          LSTM**
SVR vs RF            17.6444   0.000000            RF**
SVR vs CNN           11.9659   0.000000           CNN**
SVR vs RNN            4.5692   0.000005           RNN**
SVR vs LSTM          50.5700   0.000000          LSTM**
RF vs CNN           -11.0101   0.000000            RF**
RF vs RNN           -11.7656   0.000000            RF**
RF vs LSTM           34.1926   0.000000          LSTM**
CNN vs RNN           -0.8088   0.418649           无显著差异
CNN vs LSTM          47.4233   0.000000          LSTM**
RNN vs LSTM          41.5628  

DM检验——MAE

In [15]:
import pandas as pd
import numpy as np
from scipy import stats

models = ['ARIMA', 'SVR', 'RF', 'CNN', 'RNN', 'LSTM']

data_dict = {}
for model in models:
    filename = f'BigData_{model}_predictions.csv'
    df = pd.read_csv(filename)
    data_dict[model] = df
sample_df = data_dict[models[0]]
identifier_cols = ['date']

stock_cols = ['stock_code']
for col in stock_cols:
    if col in sample_df.columns:
        identifier_cols.append(col)
        break


for model in models:
    df = data_dict[model]
    df['_merge_key'] = df[identifier_cols].astype(str).agg('_'.join, axis=1)
    data_dict[model] = df

# 获取第一个模型的键集合
common_keys = set(data_dict[models[0]]['_merge_key'])

# 与其他模型取交集
for model in models[1:]:
    model_keys = set(data_dict[model]['_merge_key'])
    common_keys = common_keys.intersection(model_keys)

# 转换为列表
common_keys = list(common_keys)
aligned_data_dict = {}
errors_dict = {}

for model in models:
    # 筛选出共同键的数据
    df_aligned = data_dict[model][data_dict[model]['_merge_key'].isin(common_keys)].copy()
    
    # 按标识列排序以确保顺序一致
    df_aligned = df_aligned.sort_values(identifier_cols).reset_index(drop=True)
    
    # 存储对齐后的数据
    aligned_data_dict[model] = df_aligned
    
    # 计算绝对误差 (AE)
    errors_dict[model] = np.abs(df_aligned['actual_return'] - df_aligned['predicted_return']).values

# Diebold-Mariano检验函数
def dm_test(e1, e2, h=1, power=1):
    """
    Diebold-Mariano检验
    
    参数:
    e1: 模型1的预测误差(绝对误差AE)
    e2: 模型2的预测误差(绝对误差AE)
    h: 预测步长 (默认为1)
    power: 损失函数的幂次 (对于MAE使用1)
    
    返回:
    DM统计量和p值
    """
    # 计算损失差异
    d = e1**power - e2**power
    
    # 计算均值
    d_mean = np.mean(d)
    
    # 计算方差（考虑自相关）
    def autocovariance(series, lag):
        n = len(series)
        c0 = np.dot(series, series) / n
        if lag == 0:
            return c0
        else:
            c_lag = np.dot(series[:-lag], series[lag:]) / n
            return c_lag
    
    # 计算长期方差
    gamma_0 = autocovariance(d, 0)
    gamma_sum = 0
    for lag in range(1, h):
        gamma_sum += autocovariance(d, lag)
    
    d_var = (gamma_0 + 2 * gamma_sum) / len(d)
    
    # 避免除以零
    if d_var <= 0:
        return np.nan, np.nan
    
    # 计算DM统计量
    dm_stat = d_mean / np.sqrt(d_var)
    
    # 计算p值（双侧检验）
    p_value = 2 * (1 - stats.norm.cdf(np.abs(dm_stat)))
    
    return dm_stat, p_value

# 存储原始p值用于后续分析
p_values = {}
dm_stats = {}

# 对所有模型对进行DM检验
for i, model1 in enumerate(models):
    for j, model2 in enumerate(models):
        if i < j:
            # 执行DM检验（使用绝对误差，power=1）
            dm_stat, p_value = dm_test(errors_dict[model1], errors_dict[model2], power=1)
            
            # 存储原始值
            p_values[(model1, model2)] = p_value
            dm_stats[(model1, model2)] = dm_stat

print("=" * 70)
print("Diebold-Mariano检验结果表 (DM p-value):")
print("=" * 70)

detailed_results = []
for i, model1 in enumerate(models):
    for j, model2 in enumerate(models):
        if i < j:
            p_value = p_values[(model1, model2)]
            dm_stat = dm_stats[(model1, model2)]

            if p_value < 0.01:
                significance = "**"
            elif p_value < 0.05:
                significance = "*"

            # 判断哪个模型更好和结论
            if p_value < 0.05:  # 拒绝原假设
                if dm_stat > 0:  # Model1的MAE > Model2的MAE
                    better_model = model2
                else:  # Model1的MAE < Model2的MAE
                    better_model = model1
            else:  # 不能拒绝原假设
                better_model = "无显著差异"
            
            detailed_results.append({
                'Model 1': model1,
                'Model 2': model2,
                'DM Statistic': dm_stat,
                'p-value': p_value,
                'Significance': significance,
                'Better Model': better_model,
            })

detailed_df = pd.DataFrame(detailed_results)

# 打印详细结果摘要
print(f"{'比较':13s}{'DM统计量':>8s} {'p值':>10s} {'结论':>15s}")
print("-" * 85)
for _, row in detailed_df.iterrows():
    comparison = f"{row['Model 1']} vs {row['Model 2']}"
    better = row['Better Model']
    if better != '无显著差异':
        better_mark = f"{better}{row['Significance']}"
    else:
        better_mark = better
    print(f"{comparison:15s} {row['DM Statistic']:12.4f} {row['p-value']:10.6f} {better_mark:>15s}")

Diebold-Mariano检验结果表 (DM p-value):
比较              DM统计量         p值              结论
-------------------------------------------------------------------------------------
ARIMA vs SVR         43.0825   0.000000           SVR**
ARIMA vs RF          65.7944   0.000000            RF**
ARIMA vs CNN         46.9761   0.000000           CNN**
ARIMA vs RNN         91.7643   0.000000           RNN**
ARIMA vs LSTM        84.5816   0.000000          LSTM**
SVR vs RF            16.4024   0.000000            RF**
SVR vs CNN           10.2900   0.000000           CNN**
SVR vs RNN           -5.1466   0.000000           SVR**
SVR vs LSTM          57.4987   0.000000          LSTM**
RF vs CNN           -10.9946   0.000000            RF**
RF vs RNN           -23.2200   0.000000            RF**
RF vs LSTM           43.3130   0.000000          LSTM**
CNN vs RNN           -9.8171   0.000000           CNN**
CNN vs LSTM          55.9512   0.000000          LSTM**
RNN vs LSTM          58.1955   0.000000       

### 用于优化投资组合模型的数据结构展示

In [16]:
import pandas as pd
df = pd.read_csv('BigData_LSTM_predictions.csv')
df

,stock_code,date,actual_return,predicted_return,skew_20d,kurt_20d,prediction_error
0,300059,2020-01-02,0.008244,-0.002976,1.060223,1.075307,0.011219
1,300059,2020-01-03,0.006289,-0.003282,1.000763,1.039341,0.009571
2,300059,2020-01-06,0.009375,-0.003701,1.142253,1.443323,0.013076
3,300059,2020-01-07,-0.000619,-0.004068,1.088560,1.369772,0.003449
4,300059,2020-01-08,-0.037175,-0.004358,1.079849,1.446524,-0.032817
...,...,...,...,...,...,...,...
21360,600036,2024-12-09,0.000000,0.001440,-0.139766,0.327982,-0.001440
21361,600036,2024-12-10,0.023873,0.001776,-0.271223,0.692165,0.022097
21362,600036,2024-12-11,-0.007772,0.002144,-0.014369,0.851194,-0.009916
21363,600036,2024-12-12,0.013055,0.002352,0.143075,0.722167,0.010703
